In [15]:
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable, Optional
import matplotlib.pyplot as plt

In [16]:
from simulator import generate_hierarchical
from posteriors import *
from samplers_hierarchical import *

In [17]:
rng = np.random.default_rng(221)

print("Generating data")
ds = generate_hierarchical(rng=rng, truth_model = "power_law")
K = len(ds.seasons)
print(f"K = {K}, "
     f"True means = {ds.phi_k}")


Generating data
K = 10, True means = [1.23580097e-05 9.27441331e-06 8.02758220e-06 8.77680051e-06
 1.34651853e-05 1.00865273e-05 1.08705475e-05 1.33043578e-05
 6.35077821e-06 1.12390328e-05]


In [18]:
param_names = []

for i in range(K):
    param_names.append(f"log_phi_{i+1}")

param_names.extend(["gamma", "log_eta", "delta", "mu_phi", "log_sigma_phi"])
param_names

['log_phi_1',
 'log_phi_2',
 'log_phi_3',
 'log_phi_4',
 'log_phi_5',
 'log_phi_6',
 'log_phi_7',
 'log_phi_8',
 'log_phi_9',
 'log_phi_10',
 'gamma',
 'log_eta',
 'delta',
 'mu_phi',
 'log_sigma_phi']

In [19]:

true_values = {
    **{f"log_phi_{k+1}": np.log(ds.phi_k[k]) for k in range(K)},
    "gamma": ds.true_params["gamma"],
    "log_eta": np.log(ds.true_params["eta"]),
    "delta": ds.true_params["delta"],
    "mu_phi": ds.true_params["mu_phi"],
    "log_sigma_phi": np.log(ds.true_params["sigma_phi"]),
}


In [20]:
mu_phi_true = ds.true_params["mu_phi"]
sigma_phi_true = ds.true_params["sigma_phi"]

z_k_true = (np.log(ds.phi_k) - mu_phi_true) / sigma_phi_true

theta_true = np.concatenate([
    z_k_true,
    [
        ds.true_params["gamma"],
        np.log(ds.true_params["eta"]),
        ds.true_params["delta"],
        mu_phi_true,
        np.log(sigma_phi_true),
    ],
])

theta_init = theta_true + rng.normal(0, 0.05, size=K+5)

In [21]:
prior_type = "lognormal_gamma"
parameterization = "noncentered"

def log_post(theta):
    return log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

def grad_log_post(theta):
    return grad_log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

In [22]:
rwmh_results = run_multiple_chains(
        run_rwmh,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        n_iterations=100000,
        n_burnin=20000,
        adapt_until=20000,
        adapt_proposal=True,
        param_names=param_names,
    )

Iteration 5000/100000: accept rate = 0.254, scale = 0.939, elapsed = 1.0s
Iteration 10000/100000: accept rate = 0.246, scale = 0.490, elapsed = 2.1s
Iteration 15000/100000: accept rate = 0.244, scale = 0.691, elapsed = 3.1s
Iteration 20000/100000: accept rate = 0.238, scale = 0.288, elapsed = 4.2s
Iteration 25000/100000: accept rate = 0.257, scale = 0.288, elapsed = 5.2s
Iteration 30000/100000: accept rate = 0.267, scale = 0.288, elapsed = 6.3s
Iteration 35000/100000: accept rate = 0.278, scale = 0.288, elapsed = 7.3s
Iteration 40000/100000: accept rate = 0.287, scale = 0.288, elapsed = 8.3s
Iteration 45000/100000: accept rate = 0.293, scale = 0.288, elapsed = 9.4s
Iteration 50000/100000: accept rate = 0.299, scale = 0.288, elapsed = 10.4s
Iteration 55000/100000: accept rate = 0.300, scale = 0.288, elapsed = 11.4s
Iteration 60000/100000: accept rate = 0.296, scale = 0.288, elapsed = 12.6s
Iteration 65000/100000: accept rate = 0.298, scale = 0.288, elapsed = 13.6s
Iteration 70000/100000

In [23]:
rwmh_cov = estimate_dense_precond_from_rwmh(rwmh_results, ridge=1e-6)

In [24]:
mala_results = run_multiple_chains(
        run_mala,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        grad_log_posterior_fn=grad_log_post,
        n_iterations=100000,
        n_burnin=20000,
        step_size=1e-3,
        adapt_step=True,
        adapt_until=20000,
        target_accept=0.65,
        param_names=param_names,
        precond=rwmh_cov,
        adapt_precond=False,
        precond_type="dense",
        normalize_precond=True,
    )

Iteration 5000/100000: accept rate = 0.787, step_size = 0.0136311, elapsed = 2.7s
Iteration 10000/100000: accept rate = 0.707, step_size = 0.00983337, elapsed = 5.4s
Iteration 15000/100000: accept rate = 0.688, step_size = 0.0147095, elapsed = 8.0s
Iteration 20000/100000: accept rate = 0.677, step_size = 0.0159169, elapsed = 10.9s
Iteration 25000/100000: accept rate = 0.667, step_size = 0.0159169, elapsed = 13.6s
Iteration 30000/100000: accept rate = 0.664, step_size = 0.0159169, elapsed = 16.2s
Iteration 35000/100000: accept rate = 0.662, step_size = 0.0159169, elapsed = 19.1s
Iteration 40000/100000: accept rate = 0.653, step_size = 0.0159169, elapsed = 21.7s
Iteration 45000/100000: accept rate = 0.654, step_size = 0.0159169, elapsed = 24.3s
Iteration 50000/100000: accept rate = 0.652, step_size = 0.0159169, elapsed = 27.0s
Iteration 55000/100000: accept rate = 0.652, step_size = 0.0159169, elapsed = 29.6s
Iteration 60000/100000: accept rate = 0.651, step_size = 0.0159169, elapsed = 3

In [ ]:
# print_diagnostics_multi({"RWMH": rwmh_results,"MALA": mala_results}, true_values = true_values)

In [ ]:
print_diagnostics_multi_hierarchical(
    {"RWMH": rwmh_results, "MALA": mala_results},
    parameterization="noncentered",
    K=10,
    latent_display="raw",
    true_values=true_values,
)

save_traceplots_multi_hierarchical(
    mala_results,
    "traceplots_mala_noncentered_logphi.png",
    parameterization="noncentered",
    K=10,
    latent_display="log_phi",
)


save_traceplots_multi_hierarchical(
    rwmh_results,
    "traceplots_rwmh_noncentered_logphi.png",
    parameterization="noncentered",
    K=10,
    latent_display="log_phi",
)


Sampler      Chains  Accept%  Time(s)ESS(z_1)ESS(z_2)ESS(z_3)ESS(z_4)ESS(z_5)ESS(z_6)ESS(z_7)ESS(z_8)ESS(z_9)ESS(z_10)ESS(gamma)ESS(log_eta)ESS(delta)ESS(mu_phi)ESS(log_sigma_phi)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RWMH              4    0.249     82.7       2125       2441       2101       2274       1333       2458       2244       2776       1835       2093        902        497       1320        773        704
MALA              4    0.610    263.2       2199       2266       2079       2108       1833       2289       2639       2075       2231       2413        762        472       1510        957       1032

Sampler      Chains  Accept%  Time(s)Rhat(z_1)Rhat(z_2)Rhat(z_3)Rhat(z_4)Rhat(z_5)Rhat(z_6)Rhat(z_7)Rhat(z_8)Rhat(z_9)Rhat(z_10)Rhat(gamma)Rhat(log_eta)Rhat(delta)Rhat(mu_phi)Rhat(log_sigma_phi)


In [ ]:
log_posterior_hierarchical(theta_true, ds.seasons, "centered")

-27771899.857627146

In [ ]:
theta_mala = np.array([-0.1009,0.3353,0.0046,-0.0057,0.3186,-0.1673,-0.1265,
                      -0.1843,-0.1314,-0.1431,2.4964,6.2411,3.6875,1.5748,-1.4372])
log_posterior_hierarchical(theta_mala, ds.seasons, "centered")

-90240661309.22896